# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Websevel/FlyRank-Internee/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane (ML-03) framed this as classification: predict `is_declining_label` (1 when `trend_direction == "down"`), a yes/no
observed label. Per `training-honest-models`, that shape starts with **Logistic Regression** (readable — coefficients say
directly which features push the score up or down) then a **stronger method if it earns its place**. I'm training three:

- **Logistic Regression** — the readable starting point.
- **Random Forest** — handles nonlinearities/interactions without me hand-engineering them.
- **Gradient Boosting** (`HistGradientBoostingClassifier`, "where safe" per the menu — it's a standard sklearn estimator,
  no extra dependency, and cheap enough on 30k rows to run safely here) — usually the strongest of the three; if it doesn't
  beat the simpler models by a real margin, that's a finding too, not a reason to hide the simpler ones.

Features are the same numeric/categorical set `scripts/ml_utils.py` names as `MODEL_NUMERIC_FEATURES` /
`MODEL_CATEGORICAL_FEATURES` (I did not edit that file — I rebuilt the same feature list here, from the raw CSV, since
`data/processed/` isn't populated in my clone). `trend_direction` and `trend_pct` are excluded — they *are* the label
source (data dictionary, rule #2) — and I added `has_<col>` flags for the keyword/word-count columns instead of a blind
`fillna(0)`, since missingness there tracks `content_type`, not randomness.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score,
                              roc_auc_score, confusion_matrix, balanced_accuracy_score)
from sklearn.inspection import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

df = pd.read_csv("/content/content_refresh_anonymized (1).csv")
print(f"raw rows: {len(df):,}")

# Same filter scripts/01_prepare_features.py applies: needs real impressions and a full 90-day window
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)
print(f"prepared rows: {len(df):,}")

df["is_declining_label"] = df["trend_direction"].astype(str).str.lower().eq("down").astype(int)
print(f"declining rate: {df['is_declining_label'].mean():.3f}  (matches the 54.2% the data dictionary reports)")

# Missingness flags BEFORE filling — flyrank-data skill: missingness tracks content_type, don't blind-fillna it away
for col in ["search_volume", "competition", "cpc", "word_count", "char_count"]:
    df[f"has_{col}"] = df[col].notna().astype(int)

numeric_fill_zero = ["search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d", "days_with_impressions",
    "days_with_sessions", "content_age_days", "days_since_last_update", "ctr", "avg_position",
    "engagement_rate", "scroll_rate", "ai_traffic_pct"]
for c in numeric_fill_zero:
    df[c] = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)

df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

for c in ["competition_level", "content_type", "main_intent", "age_tier", "freshness_tier",
          "word_count_tier", "impression_tier", "position_tier"]:
    df[c] = df[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

NUMERIC_FEATURES = ["search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "has_search_volume", "has_competition", "has_cpc", "has_word_count", "has_char_count"]
CATEGORICAL_FEATURES = ["competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier"]

print(f"{len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical features"
      f" — trend_direction/trend_pct excluded (label source, never a feature)")

raw rows: 30,000
prepared rows: 30,000
declining rate: 0.542  (matches the 54.2% the data dictionary reports)
23 numeric + 8 categorical features — trend_direction/trend_pct excluded (label source, never a feature)


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`**, 80/20, via `GroupShuffleSplit` — same split for the baseline and every model below. The data
dictionary is explicit that `client_id` is for grouping only, and names client-holdout as the honest split for this
dataset. Content items from the same client share one site's templates, publishing cadence, and SEO practices, so a
random row-level split would let the model see other pages from the *same* client at train time and effectively
memorize client identity instead of learning a pattern that generalizes to a client it has never seen. A time-aware
split isn't available here — this CSV is one trailing-90-day snapshot per row, not a dated panel — so client-grouping is
the split that matches this data.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(df, groups=df["client_id"]))
train_df, test_df = df.iloc[train_idx].copy(), df.iloc[test_idx].copy()

print(f"train: {len(train_df):,} rows, {train_df['client_id'].nunique()} clients")
print(f"test:  {len(test_df):,} rows, {test_df['client_id'].nunique()} clients")
overlap = set(train_df["client_id"]) & set(test_df["client_id"])
print(f"client_id overlap between train and test: {len(overlap)}  (must be 0 for a true client holdout)")
print(f"train declining rate: {train_df['is_declining_label'].mean():.3f}  |  "
      f"test declining rate: {test_df['is_declining_label'].mean():.3f}")

train: 23,837 rows, 25 clients
test:  6,163 rows, 7 clients
client_id overlap between train and test: 0  (must be 0 for a true client holdout)
train declining rate: 0.550  |  test declining rate: 0.511


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*
The **Week-4 baseline** is the rule from `w04_baseline_score.ipynb`, reimplemented unchanged here: `STALE_AND_UNDERPERFORMING`
when `days_since_last_update > 180` AND `ctr` is below its `position_tier` median, `STALE_ONLY` when just stale, else
`CTR_GAP_ONLY` / `HEALTHY`. That rule was built to flag "worth refreshing," not to predict `is_declining_label` directly, so
I read it as a declining-predictor by treating the two staleness-driven reason codes (`STALE_AND_UNDERPERFORMING`,
`STALE_ONLY`) as "predicted declining" — the only part of the rule that reasons about a *trend* at all. I also add the plain
majority-class baseline my Week-3 framing said I'd show. All three baselines and all three models are scored on the exact
same `test_df` from section 2.

**Result, and the honest catch:** on raw F1-for-the-declining-class, the majority baseline looks *best* — but that's an
artifact of the label being close to balanced (51% declining in this test split): "always predict declining" gets
free recall and a misleadingly high F1 without discriminating anything. **Macro-F1 / balanced accuracy** correct for
that: there, majority and the Week-4 rule both sit at **0.50 (chance)**, while all three trained models sit around
**0.58** — a real, if modest, lift. The **ranking view (precision@300)** — closer to how this would actually get used,
as a prioritized queue — tells the same story more sharply: the Week-4 rule barely edges the base rate, while Gradient
Boosting and Logistic Regression pull meaningfully ahead. And the Week-4 rule's staleness flags are rare enough
(174 of 30,000 rows total) that **zero** of them landed in this test split's held-out clients at all — its 0.000
precision/recall row isn't the model failing to compare fairly, it's the rule not firing on unseen clients, a real
limitation worth reporting rather than hiding.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# ---- Week-4 rule baseline, reimplemented exactly as in w04_baseline_score.ipynb ----
expected_ctr = df.groupby("position_tier")["ctr"].transform("median")
df["ctr_gap"] = df["ctr"] < expected_ctr

def score_row(row, ctr_gap):
    stale = row["days_since_last_update"] > 180
    if stale and ctr_gap:
        return 90, "STALE_AND_UNDERPERFORMING"
    elif stale:
        return 55, "STALE_ONLY"
    elif ctr_gap:
        return 40, "CTR_GAP_ONLY"
    else:
        return 5, "HEALTHY"

res = [score_row(r, g) for (_, r), g in zip(df.iterrows(), df["ctr_gap"])]
df[["baseline_score", "baseline_reason"]] = pd.DataFrame(res, index=df.index)
df["baseline_predicted_declining"] = df["baseline_reason"].isin(
    ["STALE_AND_UNDERPERFORMING", "STALE_ONLY"]).astype(int)
test_df = df.loc[test_df.index].copy()  # re-attach baseline columns to the same test rows

print("Week-4 rule reason codes, whole dataset:")
print(df["baseline_reason"].value_counts())
print(f"\nstaleness-based flags in test split (the ones scored below): "
      f"{test_df['baseline_predicted_declining'].sum()} of {len(test_df)}")

X_train, y_train = train_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], train_df["is_declining_label"]
X_test, y_test = test_df[NUMERIC_FEATURES + CATEGORICAL_FEATURES], test_df["is_declining_label"]

pre = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_FEATURES),
    ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=8, class_weight="balanced",
                                             random_state=RANDOM_STATE, n_jobs=-1),
    "Gradient Boosting (Hist)": HistGradientBoostingClassifier(max_depth=4, random_state=RANDOM_STATE),
}

fitted, rows = {}, []
maj_class = y_train.mode()[0]
maj_pred = np.full(len(y_test), maj_class)
base_pred = test_df["baseline_predicted_declining"].values

def score_row_metrics(name, pred, proba=None):
    return [name, precision_score(y_test, pred, zero_division=0), recall_score(y_test, pred, zero_division=0),
            f1_score(y_test, pred, zero_division=0), accuracy_score(y_test, pred),
            f1_score(y_test, pred, average="macro"), balanced_accuracy_score(y_test, pred),
            roc_auc_score(y_test, proba) if proba is not None else np.nan]

rows.append(score_row_metrics("Majority-class baseline", maj_pred))
rows.append(score_row_metrics("Week-4 rule baseline", base_pred))

for name, clf in models.items():
    pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(X_train, y_train)
    fitted[name] = pipe
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    rows.append(score_row_metrics(name, pred, proba))

results = pd.DataFrame(rows, columns=["Method", "Precision", "Recall", "F1 (declining class)",
                                       "Accuracy", "Macro-F1", "Balanced Accuracy", "ROC-AUC"])
print("\n=== Model vs baseline — same split, same features, same test set ===")
print(results.round(3).to_string(index=False))

# Ranking view: this is a prioritized-queue product, so precision@K matters as much as classification metrics
def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(y_true)[order][:k].mean()

K = 300
print(f"\n=== Precision@{K} (ranking view; test base rate = {y_test.mean():.3f}) ===")
print(f"{'Week-4 rule score':<28} {precision_at_k(y_test.values, test_df['baseline_score'].values, K):.3f}")
for name, pipe in fitted.items():
    proba = pipe.predict_proba(X_test)[:, 1]
    print(f"{name:<28} {precision_at_k(y_test.values, proba, K):.3f}")

Week-4 rule reason codes, whole dataset:
baseline_reason
HEALTHY                      16841
CTR_GAP_ONLY                 12985
STALE_AND_UNDERPERFORMING       94
STALE_ONLY                      80
Name: count, dtype: int64

staleness-based flags in test split (the ones scored below): 0 of 6163

=== Model vs baseline — same split, same features, same test set ===
                  Method  Precision  Recall  F1 (declining class)  Accuracy  Macro-F1  Balanced Accuracy  ROC-AUC
 Majority-class baseline      0.511   1.000                 0.676     0.511     0.338              0.500      NaN
    Week-4 rule baseline      0.000   0.000                 0.000     0.489     0.328              0.500      NaN
     Logistic Regression      0.586   0.628                 0.606     0.583     0.582              0.582    0.616
           Random Forest      0.586   0.603                 0.595     0.580     0.579              0.579    0.603
Gradient Boosting (Hist)      0.584   0.661                 0.620

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*
Best model by both classification F1 and precision@300 is **Gradient Boosting**, so that's the one I read errors from.

**What it leans on** (permutation importance, scored on the same held-out clients): `days_with_impressions` and
`log_impressions_90d` dominate by a wide margin, with `ctr` a distant third and everything else close to zero. That's
plausible, not suspicious — `trend_pct` (the label's source) is an impressions ratio, so how consistently and how much
a page shows impressions at all is a reasonable, non-leaky proxy for how its traffic is trending; it isn't a
"suspiciously perfect" single feature carrying the whole signal (importance is ~0.07, not ~1.0), which is what I'd
expect if `trend_pct`/`trend_direction` had slipped in.

**Where it's wrong:** false negatives (missed decliners) cluster at very low, sparse traffic — a handful of impressions,
zero clicks, `ctr = 0` — where there just isn't enough signal in 90 days to separate "declining" from "always tiny."
False positives cluster at moderate-to-good traffic with reasonable position and non-zero CTR — pages that look healthy
on every feature I have but are still trending down, which is exactly the kind of case a stale-content *rule* would
also miss, and the reason a learned model has any edge here at all: the honest edge is modest (balanced accuracy 0.58
vs 0.50 chance), and no feature or metric here justifies claiming more than that.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

best_name = "Gradient Boosting (Hist)"
best_pipe = fitted[best_name]
pred = best_pipe.predict(X_test)
proba = best_pipe.predict_proba(X_test)[:, 1]

cm = confusion_matrix(y_test, pred)
print(f"Confusion matrix for {best_name} (rows=true 0/1, cols=predicted 0/1):")
print(cm)
tn, fp, fn, tp = cm.ravel()
print(f"TN={tn}  FP={fp}  FN={fn}  TP={tp}")

perm = permutation_importance(best_pipe, X_test, y_test, n_repeats=8, random_state=RANDOM_STATE,
                               scoring="f1", n_jobs=-1)
feat_names = NUMERIC_FEATURES + CATEGORICAL_FEATURES
imp_df = pd.DataFrame({"feature": feat_names, "importance_mean": perm.importances_mean,
                        "importance_std": perm.importances_std}).sort_values("importance_mean", ascending=False)
print("\nTop 8 features by permutation importance (drop in F1 when shuffled):")
print(imp_df.head(8).to_string(index=False))

test_df = test_df.copy()
test_df["pred"] = pred
test_df["proba"] = proba
fn_rows = test_df[(test_df["is_declining_label"] == 1) & (test_df["pred"] == 0)]
fp_rows = test_df[(test_df["is_declining_label"] == 0) & (test_df["pred"] == 1)]
cols = ["content_id", "avg_position", "ctr", "impressions_90d", "days_since_last_update", "proba"]

print(f"\n3 false negatives (missed decliners) — {len(fn_rows)} total in test:")
print(fn_rows[cols].head(3).to_string(index=False))
print(f"\n3 false positives (flagged, actually fine) — {len(fp_rows)} total in test:")
print(fp_rows[cols].head(3).to_string(index=False))

Confusion matrix for Gradient Boosting (Hist) (rows=true 0/1, cols=predicted 0/1):
[[1533 1481]
 [1068 2081]]
TN=1533  FP=1481  FN=1068  TP=2081

Top 8 features by permutation importance (drop in F1 when shuffled):
              feature  importance_mean  importance_std
days_with_impressions         0.070905        0.003965
  log_impressions_90d         0.039611        0.002727
                  ctr         0.001979        0.001133
     log_sessions_90d         0.000815        0.000806
        search_volume         0.000589        0.001430
       log_clicks_90d         0.000520        0.000760
             age_tier         0.000431        0.000303
          main_intent         0.000374        0.000535

3 false negatives (missed decliners) — 1068 total in test:
          content_id  avg_position  ctr  impressions_90d  days_since_last_update    proba
content_4595e8704e07          36.3  0.0                4                     104 0.169467
content_40cb4af260c0          25.5  0.0           

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.